# 데모 큰 흐름

1. 해양수산부 PORT-MIS에서 26년 6월 21일 부산항 '선박입출항현황' 데이터 불러오기 

2. delta table 에 매핑되도록 컬럼명 제약 제거 후 bronze 저장

3. 컬럼명 소문자로 재수정/ 날짜타입 변경 / stay_days 컬럼 추가 후 silver 저장

4. 품질 규칙(필수 컬럼 누락 비율, 체류시간 비율, 중복된 데이터 비율 등) 만들기

5. 가드레일을 위한 품질 등급 나누기

6. 품질 항목별 gold 저장

7. 가드레일 확인 

# 추가할 것

1. '부두별관제현황', '통과선박FAX확인' 등의 데이터 추가하여 bronze 적재 - 데이터타입 및 스키마 맞춰서 silver 적재 - 품질 KPI 별 gold 적재

2. '이상치'를 어떻게 정의할 것인가.

### 1. 데이터 로드

 spark로 바로 읽으니 컬럼매핑에 어려움이 있어서 pandas로 읽은 후 spark로 저장함.


In [0]:
import pandas as pd

local_path = "file:/Workspace/Users/4dt035@msacademy.msai.kr/portmis_arrival_demo(sheet).csv"

pdf = pd.read_csv(
    local_path,
    skiprows=11,
    encoding="utf-8"
)

df_raw = spark.createDataFrame(pdf)

display(df_raw)

In [0]:
# ========================
# 참고. 로드 실패 했던 셀
# =========================
# file_path ="file:/Workspace/Users/4dt035@msacademy.msai.kr/portmis_arrival_demo(sheet).csv"

# df_raw = (
#     spark.read
#     .option("header", "true")
#     .option("encoding", "UTF-8")
#     .option("multuline", "true")
#     .option("skipRows", 13)
#     .csv(file_path)
# )

# display(df_raw)

# 행은 잘 읽으나 컬럼을 읽지 못함

### 2. 컬럼명 제약 제거 후 bronze로 저장

In [0]:
# ===================
# bronze table
# 1번째 시도에서 컬럼명제약으로 인해 bronze 저장이 불가능했음
# 2번째 시도에서 컬럼명제약을 제거하여 성공
# ===================

import re
from collections import Counter

def clean_column_name(col_name):
    '''
    Delta Lake에 적재하기 위해서는 컬럼명에 다음 규칙을 따르도록 합니다.
    '''
    # 앞 뒤 공백 제거
    name = str(col_name).strip()

    # 줄바꿈, 탭을 언더바로 변경
    name = re.sub(r"[\n\t\r]+", "_", name)

    # 특수 문자 언더바로 변경 
    name = re.sub(r"[ ,;{}()\n\t=]+", "_", name)
    name = name.replace(":", "_")

    # 연속 언더바, 앞뒤 언더바 정리
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    
    if name == "":
        name = "unknown_column"
    
    return name

def make_unique_columns(cols):
    '''
    컬럼명 중복을 방지합니다.
    '''
    counts = Counter()
    unique_cols = []
    
    for c in cols:
        counts[c] += 1
        if counts[c] == 1:
            unique_cols.append(c)
        else:
            unique_cols.append(f"{c}_{counts[c]}")
    
    return unique_cols

old_cols = df_raw.columns
clean_cols = [clean_column_name(c) for c in old_cols]
new_cols = make_unique_columns(clean_cols)

for old, new in zip(old_cols, new_cols):
    print(f"{old}  ->  {new}")

# 컬럼명 변경
df_bronze = df_raw.toDF(*new_cols)

mapping_data = list(zip(old_cols, new_cols))
df_column_mapping = spark.createDataFrame(
    mapping_data,
    ["original_column_name", "bronze_column_name"]
)

# 원본 컬럼명 bronze table 저장
df_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "bronze_portmis_arrival_raw"
)

# 수정된 컬럼명 bronze table은 아래와 같이 저장
df_column_mapping.write.format("delta").mode("overwrite").saveAsTable(
    "meta_portmis_arrival_column_mapping"
)

### 3. 컬럼명 변경 후 silver table 적재

In [0]:
df_raw.columns

[컬럼명 뜻 간단요약]

항명: 항구이름

호출부호, 선명: 선박이름과 코드

입항횟수: 해당년도

입항횟수.1: 해당년도에 해당항구에 입항한 횟수

구분: ?

외내: 외항선/내항선 구분

CIQ 수속일자: 세관/출입국/검역 수속 진행된 일시

수리일시: ?

MRN 번호: ?

계선장소, 계선장소.1, 계선장소.2: 항구 및 포트번호



In [0]:
# ====================================
# 컬럼명정리 및 stay_days 컬럼 추가하여 silver 적재
# df_silver_1try
# ====================================

from pyspark.sql.functions import col, to_timestamp, datediff, unix_timestamp, when, count, isnan

df_silver_to_1try= df_raw.select(
    col("항명").alias("port_name"),
    col("호출부호").alias("call_sign"),
    col("선명").alias("vessel_name"),
    to_date(to_timestamp(col("입항일시"), "yyyy-MM-dd HH:mm")).alias("base_date"),
    to_timestamp(col("입항일시"), "yyyy-MM-dd HH:mm").alias("arrival_datetime"),
    to_timestamp(col("출항일시"), "yyyy-MM-dd HH:mm").alias("departure_datetime"),
    to_timestamp(col("CIQ수속일자")).alias("CIQ"),
    col("계선장소").alias("berth_place"),
    col("선박용도").alias("vessel_type")
)

# stay_days는 해당 항구에 체류한 일수 = (출항일시 - 입항일시)
# 다만 이름은 발표/코드상으로는 stay_days 또는 turnaround_days가 더 좋아 보여.
from pyspark.sql.functions import round
df_silver_1try = df_silver_to_1try.withColumn( "stay_days", round((unix_timestamp("departure_datetime") - unix_timestamp("arrival_datetime")) / (60*60*24), 2))


display(df_silver_1try)

PORT-MIS 선박입출항현황에는 ETA가 없어 예정 대비 지연시간은 계산하지 못했습니다. 대신 공개 데이터에 포함된 입항일시와 출항일시를 활용해 항만 체류시간을 계산했고, 출항이 입항보다 빠른 오류, 시간 파싱 실패, 비정상 장기 체류, 중복 이벤트를 품질 규칙으로 점검했습니다. 최종적으로 이 품질점수에 따라 Text-to-SQL 질의를 허용·경고·차단하는 구조를 데모했습니다.

### 4. 품질 규칙 만들기

In [0]:
# ====================================
# [1] 필수 컬럼 결측치 비율여부 점검
# 추후에는 vessel size등 다양한 정보 넣기
# baseline quality_df
# ====================================
required_cols = ["call_sign", "vessel_name",  "arrival_datetime", "departure_datetime", "stay_days", "berth_place"]

total_count = df_silver_1try.count()

quality_checks = []

for c in required_cols:
    null_count = df_silver_1try.filter(col(c).isNull()).count()
    null_ratio = null_count / total_count if total_count > 0 else 0
    
    quality_checks.append((c, null_count, null_ratio))

quality_df = spark.createDataFrame(
    quality_checks,
    ["column_name", "null_count", "null_ratio"]
)

display(quality_df)

In [0]:
# ===============================================
# [2] stay_days 기준 이상 체류 후보
# 도메인 기준이 확정되지 않은 상태에서는 임의 기준보다 데이터 분포 기반 P95를 이상 체류 후보로 설정
# df_issue
# ===============================================
from pyspark.sql.functions import when

p95 = df_silver_1try.approxQuantile("stay_days", [0.95], 0.01)[0]

df_issue = df_silver_1try.withColumn(
    "quality_issue",
    when(col("arrival_datetime").isNull(), "ARRIVAL_TIME_PARSE_ERROR")
    .when(col("departure_datetime").isNull(), "DEPARTURE_TIME_PARSE_ERROR")
    .when(col("stay_days") < 0, "DEPARTURE_BEFORE_ARRIVAL")
    .when(col("stay_days") > p95, "LONG_STAY")
    .otherwise("OK")
)

display(df_issue)
# LONG_STAY는 데이터품질 이슈로 인한 오류가 아닌, 운영 검토가 필요한 이상 체류 후보로 분류하였습니다

In [0]:
# ====================================
# [3] 필수 컬럼 결측률 평균 , 시간/체류기간 이슈 비율 , 중복된 데이터 비율을 합해 품질 점수 계산
# quality_df
# ====================================

from pyspark.sql.functions import avg

total_count = df_issue.count()

# ----------------------
# 필수 컬럼 결측률 평균
# ----------------------
quality_checks = []

for c in required_cols:
    null_count = df_issue.filter(col(c).isNull()).count()
    null_ratio = null_count / total_count if total_count > 0 else 0
    quality_checks.append((c, null_count, null_ratio))

quality_df = spark.createDataFrame(
    quality_checks,
    ["column_name", "null_count", "null_ratio"]
)

avg_null_ratio = quality_df.select(avg("null_ratio")).collect()[0][0]

# ----------------------
# 시간/체류기간 이슈 비율
#------------------------
# 오랜기간 체류하는 선박에 대해 명백한데이터 오류(시간 파싱 실패 등)인지, 운영 상 검토가 필요한지는 확인을 반드시 해야합니다.
issue_count = df_issue.filter(col("quality_issue") != "OK").count()
issue_ratio = issue_count / total_count if total_count > 0 else 0

# ----------------------
# 중복 비율
# ----------------------
dup_count = total_count - df_issue.dropDuplicates(
    ["vessel_name", "arrival_datetime", "departure_datetime", "berth_place"]
).count()
dup_ratio = dup_count / total_count if total_count > 0 else 0

null_penalty = avg_null_ratio * 40
issue_penalty = issue_ratio * 40
dup_penalty = dup_ratio * 20

quality_score = 100 - (null_penalty + issue_penalty + dup_penalty)
quality_score = max(0, min(100, quality_score))

print(f"필수 컬럼 결측 감점: {avg_null_ratio:.2%}")
print(f"시간/체류기간 이슈 감점: {issue_ratio:.2%}")
print(f"중복 감점: {dup_ratio:.2%}")
print(f"최종품질점수: {quality_score:.2f}")

### 5. 가드레일을 위한 품질등급

In [0]:
if quality_score >= 90:
    grade = "A"
    guardrail_status = "ALLOW"
    guardrail_message = "AI 질의 및 리포트 생성 가능"
elif quality_score >= 75:
    grade = "B"
    guardrail_status = "ALLOW_WITH_WARNING"
    guardrail_message = "일부 품질 이슈가 있어 경고 표시 후 질의 가능"
elif quality_score >= 60:
    grade = "C"
    guardrail_status = "WARNING"
    guardrail_message = "품질 보완 권장. 운영 판단용 사용 주의"
else:
    grade = "D"
    guardrail_status = "BLOCK"
    guardrail_message = "데이터 품질이 낮아 AI 질의 및 리포트 생성 차단"

print(f"품질등급: {grade}")
print(f"Text-to-SQL 질의 상태: {guardrail_message}")

### 6. 품질 유형별 gold 테이블 만들기

------------------------------------------------------
gold_portmis_quality_summary
→ 전체 데이터셋 품질점수와 가드레일 상태

gold_portmis_issue_summary
→ 품질 이슈별 건수와 비율

gold_portmis_query_ready
→ 품질 이슈가 없는 안전 조회용 데이터

gold_portmis_review_target
→ 운영자 확인이 필요한 이상 후보 데이터
---------------------------------------------------------

[1] 품질점수/등급/가드레일 판정

In [0]:

# ===========================
# [1] 품질점수/등급/가드레일 판정
# gold_portmis_quality_summary
# ============================

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

summary_data = [
    Row(
        dataset_name="PORT-MIS 선박입출항현황",

        # 이번 데모에서는 total_count를 20개로 둠.
        total_count=total_count, 

        avg_null_ratio=float(avg_null_ratio),
        issue_ratio=float(issue_ratio),
        dup_ratio=float(dup_ratio),
        quality_score=float(quality_score),
        quality_grade=grade,
        guardrail_status=guardrail_status,
        guardrail_message=guardrail_message
    )
]

gold_quality_summary = spark.createDataFrame(summary_data) \
    .withColumn("created_at", current_timestamp())

display(gold_quality_summary)

In [0]:

gold_quality_summary.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_portmis_quality_summary")

[2] 이슈 유형별 건수/비율

In [0]:
# ===========================
# [2] 이슈 유형별 건수/비율
# gold_portmis_issue_summary
# ============================

from pyspark.sql.functions import count, round

gold_issue_summary = (
    df_issue
    .groupBy("quality_issue")
    .agg(count("*").alias("issue_count"))
    .withColumn("issue_ratio", round(col("issue_count") / total_count, 4))
    .orderBy(col("issue_count").desc())
)

display(gold_issue_summary)

[4] 운영자 검토 대상 테이블

In [0]:
gold_portmis_review_target.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_portmis_review_target")

### 7. Gold 기반 “가드레일 질의 데모

질문:“항만 체류시간이 가장 긴 선박 5개 알려줘.”

In [0]:
summary = spark.table("gold_portmis_quality_summary").collect()[0]

guardrail_status = summary["guardrail_status"]
guardrail_message = summary["guardrail_message"]

print(f"가드레일 상태: {guardrail_status}")
print(f"안내 메시지: {guardrail_message}")

if guardrail_status == "BLOCK":
    print("질의 차단: 데이터 품질이 낮아 항만 체류시간 분석을 제공하지 않습니다.")
else:
    if guardrail_status in ["ALLOW_WITH_WARNING", "WARNING"]:
        print("주의: 일부 품질 이슈가 있어 결과 해석에 주의가 필요합니다.")
    
    display(
        # 운영판단에 즉각 판단할 수 있는 gold 테이블
        spark.table("gold_portmis_query_ready") 
        .orderBy(col("stay_days").desc())
        .select(
            "vessel_name",
            "arrival_datetime",
            "departure_datetime",
            "stay_days",
            "berth_place"
        )
        .limit(5)
    )

In [0]:
tables = [
    "bronze_portmis_arrival_raw",
    "meta_portmis_arrival_column_mapping",
    "gold_portmis_quality_summary",
    "gold_portmis_issue_summary",
    "gold_portmis_query_ready",
    "gold_portmis_review_target"
]

for table in tables:
    df = spark.table(table)
    output_path = f"/FileStore/exports/{table}"
    
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)
    print(f"saved: {output_path}")

In [0]:
src_dir = "/FileStore/exports/gold_portmis_quality_summary"
dst_file = "/FileStore/exports/gold_portmis_quality_summary.csv"

part_file = [f.path for f in dbutils.fs.ls(src_dir) if f.name.startswith("part-")][0]

dbutils.fs.cp(part_file, dst_file)

print("download path:")
print("/files/exports/gold_portmis_quality_summary.csv")

In [0]:
tables = [
    "gold_portmis_quality_summary",
    "gold_portmis_issue_summary",
    "gold_portmis_query_ready",
    "gold_portmis_review_target",
    "meta_portmis_arrival_column_mapping"
]

for table in tables:
    temp_dir = f"/FileStore/exports/{table}_temp"
    final_file = f"/FileStore/exports/{table}.csv"
    
    df = spark.table(table)
    
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(temp_dir)
    
    part_file = [f.path for f in dbutils.fs.ls(temp_dir) if f.name.startswith("part-")][0]
    dbutils.fs.cp(part_file, final_file)
    
    print(f"{table}: /files/exports/{table}.csv")